# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [131]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [132]:
# get file path
file_path="./data/AviationData.csv"

# load data
df =pd.read_csv(file_path,encoding="latin_1")

# get first five rows
df.head()

/var/folders/_r/g1b4b9xn2nn0xtb2011g11v80000gp/T/ipykernel_80273/421208516.py:5: DtypeWarning: Columns (0: Latitude, 1: Longitude, 2: Broad.phase.of.flight) have mixed types. Specify dtype option on import or set low_memory=False.
  df =pd.read_csv(file_path,encoding="latin_1")


,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
0,20001218X45444,Accident,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,UNK,Cruise,Probable Cause,NaN
1,20001218X45447,Accident,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,4.0,0.0,0.0,0.0,UNK,Unknown,Probable Cause,19-09-1996
2,20061025X01555,Accident,NYC07LA005,1974-08-30,"Saltville, VA",United States,36.922223,-81.878056,NaN,NaN,...,Personal,NaN,3.0,NaN,NaN,NaN,IMC,Cruise,Probable Cause,26-02-2007
3,20001218X45448,Accident,LAX96LA321,1977-06-19,"EUREKA, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,IMC,Cruise,Probable Cause,12-09-2000
4,20041105X01764,Accident,CHI79FA064,1979-08-02,"Canton, OH",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,1.0,2.0,NaN,0.0,VMC,Approach,Probable Cause,16-04-1980


In [133]:
# get missing values per column
df.isna().sum()

Event.Id                      0
Investigation.Type            0
Accident.Number               0
Event.Date                    0
Location                     52
Country                     226
Latitude                  54507
Longitude                 54516
Airport.Code              38757
Airport.Name              36185
Injury.Severity            1000
Aircraft.damage            3194
Aircraft.Category         56602
Registration.Number        1382
Make                         63
Model                        92
Amateur.Built               102
Number.of.Engines          6084
Engine.Type                7096
FAR.Description           56866
Schedule                  76307
Purpose.of.flight          6192
Air.carrier               72241
Total.Fatal.Injuries      11401
Total.Serious.Injuries    12510
Total.Minor.Injuries      11933
Total.Uninjured            5912
Weather.Condition          4492
Broad.phase.of.flight     27165
Report.Status              6384
Publication.Date          13771
dtype: i

In [134]:
# get column data types
df.dtypes

Event.Id                      str
Investigation.Type            str
Accident.Number               str
Event.Date                    str
Location                      str
Country                       str
Latitude                   object
Longitude                  object
Airport.Code                  str
Airport.Name                  str
Injury.Severity               str
Aircraft.damage               str
Aircraft.Category             str
Registration.Number           str
Make                          str
Model                         str
Amateur.Built                 str
Number.of.Engines         float64
Engine.Type                   str
FAR.Description               str
Schedule                      str
Purpose.of.flight             str
Air.carrier                   str
Total.Fatal.Injuries      float64
Total.Serious.Injuries    float64
Total.Minor.Injuries      float64
Total.Uninjured           float64
Weather.Condition             str
Broad.phase.of.flight         str
Report.Status 

In [135]:
# get statistical summary
df.describe()

,Number.of.Engines,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
count,82805.000000,77488.000000,76379.000000,76956.000000,82977.000000
mean,1.146585,0.647855,0.279881,0.357061,5.325440
std,0.446510,5.485960,1.544084,2.235625,27.913634
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.000000,1.000000
75%,1.000000,0.000000,0.000000,0.000000,2.000000
max,8.000000,349.000000,161.000000,380.000000,699.000000


## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [136]:
df =df[(df["Event.Date"]>"1982-12-31") & (df["Investigation.Type"]=="Accident")]

### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [137]:
# cols that represent injury or destruction
injury_des_cols=["Total.Fatal.Injuries",
"Total.Serious.Injuries",
"Total.Minor.Injuries",
"Total.Uninjured","Injury.Severity","Aircraft.damage"
]
# get first five rows
df[injury_des_cols].head()

,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Injury.Severity,Aircraft.damage
3600,0.0,1.0,0.0,1.0,Non-Fatal,NaN
3601,0.0,0.0,1.0,3.0,Non-Fatal,Substantial
3602,0.0,0.0,0.0,2.0,Non-Fatal,Substantial
3603,0.0,0.0,0.0,1.0,Non-Fatal,Substantial
3604,0.0,0.0,2.0,0.0,Non-Fatal,Substantial


In [138]:
# get unique values in these columns
df["Total.Fatal.Injuries"].unique()
df["Total.Serious.Injuries"].unique()
df["Total.Minor.Injuries"].unique()
df["Total.Uninjured"].unique()
df["Injury.Severity"].unique()


<StringArray>
[ 'Non-Fatal',   'Fatal(2)',   'Fatal(3)',   'Fatal(1)',   'Fatal(5)',
   'Fatal(7)',   'Fatal(6)',   'Fatal(4)',  'Fatal(23)',   'Fatal(8)',
 ...
 'Fatal(154)',  'Fatal(96)', 'Fatal(114)', 'Fatal(199)',  'Fatal(89)',
  'Fatal(57)',      'Fatal',          nan,      'Minor',    'Serious']
Length: 107, dtype: str

In [139]:
# statistical summary of injury and destruction column
df[injury_des_cols].describe()


,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
count,70799.000000,69696.000000,70249.000000,75775.000000
mean,0.686154,0.291882,0.361101,3.291917
std,5.696489,1.561689,1.793699,20.136910
min,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,1.000000
75%,0.000000,0.000000,0.000000,2.000000
max,349.000000,161.000000,200.000000,699.000000


### Decision
Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured based  on  statistical summary are heavy skewed so median imputation would label unknown cases as no injuries  and bias the data  thus decided to remove nan as nan


In [140]:
# calculate total people on the aircraft
df["Total.People"]=df["Total.Uninjured"]+df["Total.Serious.Injuries"]+df["Total.Fatal.Injuries"]+df["Total.Minor.Injuries"]

df.head()

,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date,Total.People
3600,20001214X42040,Accident,LAX83LA093,1983-01-01,"ARROYO GRANDE, CA",United States,NaN,NaN,NaN,NaN,...,NaN,0.0,1.0,0.0,1.0,VMC,Landing,Probable Cause,NaN,2.0
3601,20001214X42095,Accident,SEA83LA036,1983-01-01,"NEWPORT, OR",United States,NaN,NaN,ONP,NEWPORT MUNICIPAL,...,NaN,0.0,0.0,1.0,3.0,VMC,Approach,Probable Cause,NaN,4.0
3602,20001214X42067,Accident,MKC83LA056,1983-01-01,"WOODBINE, IA",United States,NaN,NaN,3YR,MUNICIPAL,...,NaN,0.0,0.0,0.0,2.0,VMC,Landing,Probable Cause,NaN,2.0
3603,20001214X42063,Accident,MKC83LA050,1983-01-01,"MARYVILLE, MO",United States,NaN,NaN,78Y,RANKIN,...,NaN,0.0,0.0,0.0,1.0,VMC,Takeoff,Probable Cause,NaN,1.0
3604,20001214X42018,Accident,LAX83FUG11,1983-01-01,"UPLAND, CA",United States,NaN,NaN,CCB,CABLE,...,NaN,0.0,0.0,2.0,0.0,VMC,Approach,Probable Cause,NaN,2.0


**Handling missing injury counts**

The `Total.People` column is calculated by adding the fatal, serious, minor, and uninjured counts for each accident. If any of these values is missing (`NaN`), the total will also be `NaN`. Consequently, the injury percentage cannot be calculated for that accident because the total number of people involved is unknown. I will retain these missing values rather than assume that they represent zero.

In [141]:
# calculate percentage of serious and fatal injuries to total
df["pct_fatal_and_serious"]=(df["Total.Serious.Injuries"]+df["Total.Fatal.Injuries"]) *100/df["Total.People"]


**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

Aircraft.damage column has both unknown and nan.  I replaced 'Unknown' with NaN to standardize missing data

In [142]:
# check for unique values in aircraft.damage
df["Aircraft.damage"].unique()

<StringArray>
[nan, 'Substantial', 'Destroyed', 'Minor', 'Unknown']
Length: 5, dtype: str

In [143]:
df["Aircraft.damage"] =(df["Aircraft.damage"].str.strip().str.title().replace("Unknown",np.nan))

In [144]:

destroyed_map={ "Destroyed": 1,
    "Substantial": 0,
    "Minor": 0}

# create a column to check if aircraft is destroyed or not
df["is_destroyed"]=df["Aircraft.damage"].map(destroyed_map)


is_destroyed column is create where destroyed is mapped to 1 while substancial and minor  are counted as not destroyed thus represented as 0 and nan remain as nan

### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [145]:

# general display make column to get an idea how it look like
df["Make"]

3600                        Piccard
3601                         Cessna
3602                         Cessna
3603                         Cessna
3604                          Piper
                    ...            
88884                         PIPER
88885                      BELLANCA
88886    AMERICAN CHAMPION AIRCRAFT
88887                        CESSNA
88888                         PIPER
Name: Make, Length: 81534, dtype: str

In [146]:
# get unique values
df["Make"].unique()

<StringArray>
[               'Piccard',                 'Cessna',                  'Piper',
          'Balloon Works',         'North American',                  'Beech',
             'Swearingen',               'Canadair',       'Javelin Aircraft',
                'Douglas',
 ...
    'MULHOLLAND ROBERT A',       'DETRICK DONALD G',          'SEACE DAVID A',
 'DANIEL ANTHONY PROCTOR',      'CHILDS MICHAEL  A',                'PHANTOM',
             'GREG HOBBS',      'JAMES R DERNOVSEK',          'ORLICAN S R O',
          'ROYSE RALPH L']
Length: 8062, dtype: str

In [147]:
# display data type
df["Make"].dtype

<StringDtype(storage='python', na_value=nan)>

In [148]:
df["Make"].unique()

<StringArray>
[               'Piccard',                 'Cessna',                  'Piper',
          'Balloon Works',         'North American',                  'Beech',
             'Swearingen',               'Canadair',       'Javelin Aircraft',
                'Douglas',
 ...
    'MULHOLLAND ROBERT A',       'DETRICK DONALD G',          'SEACE DAVID A',
 'DANIEL ANTHONY PROCTOR',      'CHILDS MICHAEL  A',                'PHANTOM',
             'GREG HOBBS',      'JAMES R DERNOVSEK',          'ORLICAN S R O',
          'ROYSE RALPH L']
Length: 8062, dtype: str

#### cleaning task
1. standardize the casing there is mixture of upper,lower,title

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [149]:
# standardize the casing and remove white spaces
df["Make"]=df["Make"].str.strip().str.title()

In [150]:
# get total of missing rows
df["Model"].isna().sum()

np.int64(46)

In [151]:
# drop rows with missing model
df.dropna(subset=("Model"),inplace=True)


In [152]:
# counts for each Make and Model, use
df.groupby(["Make", "Model"]).size().sort_values(ascending=False)

Make                        Model     
Cessna                      152           2217
                            172           1645
                            172N          1094
Piper                       PA-28-140      868
Cessna                      172M           755
                                          ... 
Experimental Amat Built     KITFOX III       1
Experimental Amateur Built  OWL RACER        1
Explorer Aeronautique Inc.  Ecoflyer         1
Explorer Post 51            AA CUBETTE       1
Zwicker Murray R            GLASTAR          1
Length: 17416, dtype: int64

#### Conclusion
Model labels are not unique to each make

In [154]:
# create plane.type column that unique indentify plane base on make and model
df["Plane.Type"]=df['Make'] + ' ' + df['Model']
df["Plane.Type"]
df

,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date,Total.People,pct_fatal_and_serious,is_destroyed,Plane.Type
3600,20001214X42040,Accident,LAX83LA093,1983-01-01,"ARROYO GRANDE, CA",United States,NaN,NaN,NaN,NaN,...,0.0,1.0,VMC,Landing,Probable Cause,NaN,2.0,50.0,NaN,Piccard AX-6
3601,20001214X42095,Accident,SEA83LA036,1983-01-01,"NEWPORT, OR",United States,NaN,NaN,ONP,NEWPORT MUNICIPAL,...,1.0,3.0,VMC,Approach,Probable Cause,NaN,4.0,0.0,0.0,Cessna 182P
3602,20001214X42067,Accident,MKC83LA056,1983-01-01,"WOODBINE, IA",United States,NaN,NaN,3YR,MUNICIPAL,...,0.0,2.0,VMC,Landing,Probable Cause,NaN,2.0,0.0,0.0,Cessna 182RG
3603,20001214X42063,Accident,MKC83LA050,1983-01-01,"MARYVILLE, MO",United States,NaN,NaN,78Y,RANKIN,...,0.0,1.0,VMC,Takeoff,Probable Cause,NaN,1.0,0.0,0.0,Cessna 182P
3604,20001214X42018,Accident,LAX83FUG11,1983-01-01,"UPLAND, CA",United States,NaN,NaN,CCB,CABLE,...,2.0,0.0,VMC,Approach,Probable Cause,NaN,2.0,0.0,0.0,Piper PA-28R-200
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
88884,20221227106491,Accident,ERA23LA093,2022-12-26,"Annapolis, MD",United States,NaN,NaN,NaN,NaN,...,0.0,0.0,NaN,NaN,NaN,29-12-2022,1.0,100.0,NaN,Piper PA-28-151
88885,20221227106494,Accident,ERA23LA095,2022-12-26,"Hampton, NH",United States,NaN,NaN,NaN,NaN,...,0.0,0.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,Bellanca 7ECA
88886,20221227106497,Accident,WPR23LA075,2022-12-26,"Payson, AZ",United States,341525N,1112021W,PAN,PAYSON,...,0.0,1.0,VMC,NaN,NaN,27-12-2022,1.0,0.0,0.0,American Champion Aircraft 8GCBC
88887,20221227106498,Accident,WPR23LA076,2022-12-26,"Morgan, UT",United States,NaN,NaN,NaN,NaN,...,0.0,0.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,Cessna 210N


### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [157]:
cols=[
"Engine.Type",
"Weather.Condition",
"Number.of.Engines",

"Purpose.of.flight",
"Broad.phase.of.flight"]

In [159]:
# display first five columns
df[cols].head(10)

,Engine.Type,Weather.Condition,Number.of.Engines,Purpose.of.flight,Broad.phase.of.flight
3600,Unknown,VMC,NaN,Personal,Landing
3601,Reciprocating,VMC,1.0,Personal,Approach
3602,Reciprocating,VMC,1.0,Personal,Landing
3603,Reciprocating,VMC,1.0,Personal,Takeoff
3604,Reciprocating,VMC,1.0,Personal,Approach
3605,Reciprocating,VMC,1.0,Instructional,Landing
3606,Unknown,VMC,0.0,Personal,Landing
3607,Reciprocating,VMC,2.0,Personal,Takeoff
3608,Reciprocating,VMC,1.0,Personal,Maneuvering
3609,Reciprocating,IMC,1.0,Personal,Cruise


In [160]:
# get summary of the cols
df[cols].info()

<class 'pandas.DataFrame'>
Index: 81488 entries, 3600 to 88888
Data columns (total 5 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Engine.Type            75661 non-null  str    
 1   Weather.Condition      78381 non-null  str    
 2   Number.of.Engines      76613 non-null  float64
 3   Purpose.of.flight      77185 non-null  str    
 4   Broad.phase.of.flight  56313 non-null  str    
dtypes: float64(1), str(4)
memory usage: 3.7 MB


In [167]:
# get unique engine type
df["Engine.Type"].unique()

<StringArray>
[            nan, 'Reciprocating',    'Turbo Prop',     'Turbo Fan',
   'Turbo Shaft',     'Turbo Jet',      'Electric', 'Hybrid Rocket',
            'Lr']
Length: 9, dtype: str

In [ ]:
# standardize the engine type column
df["Engine.Type"]=df["Engine.Type"].str.strip().str.title().replace({'Unknown':np.nan,'None':np.nan,'Unk':np.nan})

standardize Engine.Type column by replacing unknown,unk,none  to nan

In [171]:
# get unique weather condition
df["Weather.Condition"].unique()

<StringArray>
['VMC', 'IMC', nan]
Length: 3, dtype: str

In [ ]:
# standardize the weather condition column
df["Weather.Condition"]=df["Weather.Condition"].str.strip().str.upper().replace({'UNK':np.nan})

standardized Weather.condition column by replacing unk to nan

In [173]:
# get unique number of engines condition
df["Number.of.Engines"].unique()

array([nan,  1.,  0.,  2.,  3.,  4.,  8.,  6.])

In [ ]:
# get unique values purpose of flight condition
df["Purpose.of.flight"].unique()

<StringArray>
[                 'Personal',             'Instructional',
                  'Business',       'Executive/corporate',
                   'Unknown',        'Aerial Observation',
           'Public Aircraft',        'Aerial Application',
                     'Ferry',                         nan,
            'Other Work Use',               'Positioning',
                 'Skydiving',               'Flight Test',
             'Air Race/show',                  'Air Drop',
 'Public Aircraft - Federal',                'Glider Tow',
   'Public Aircraft - Local',             'External Load',
   'Public Aircraft - State',                'Banner Tow',
              'Firefighting',             'Air Race show',
                      'PUBS',                      'ASHO',
                      'PUBL']
Length: 27, dtype: str

In [ ]:
# standardize purpose of flight
df["Purpose.of.flight"]=df["Purpose.of.flight"].str.strip().replace('Unknown',np.nan)

In [182]:
df["Broad.phase.of.flight"].unique()

<StringArray>
[    'Landing',    'Approach',     'Takeoff', 'Maneuvering',      'Cruise',
       'Climb',   'Go-around',        'Taxi',     'Descent',           nan,
    'Standing',       'Other']
Length: 12, dtype: str

In [183]:
#standardize broad phase of flight
df["Broad.phase.of.flight"]=df["Broad.phase.of.flight"].str.strip().replace('Unknown',np.nan)

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [184]:
df.head()

,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date,Total.People,pct_fatal_and_serious,is_destroyed,Plane.Type
3600,20001214X42040,Accident,LAX83LA093,1983-01-01,"ARROYO GRANDE, CA",United States,NaN,NaN,NaN,NaN,...,0.0,1.0,VMC,Landing,Probable Cause,NaN,2.0,50.0,NaN,Piccard AX-6
3601,20001214X42095,Accident,SEA83LA036,1983-01-01,"NEWPORT, OR",United States,NaN,NaN,ONP,NEWPORT MUNICIPAL,...,1.0,3.0,VMC,Approach,Probable Cause,NaN,4.0,0.0,0.0,Cessna 182P
3602,20001214X42067,Accident,MKC83LA056,1983-01-01,"WOODBINE, IA",United States,NaN,NaN,3YR,MUNICIPAL,...,0.0,2.0,VMC,Landing,Probable Cause,NaN,2.0,0.0,0.0,Cessna 182RG
3603,20001214X42063,Accident,MKC83LA050,1983-01-01,"MARYVILLE, MO",United States,NaN,NaN,78Y,RANKIN,...,0.0,1.0,VMC,Takeoff,Probable Cause,NaN,1.0,0.0,0.0,Cessna 182P
3604,20001214X42018,Accident,LAX83FUG11,1983-01-01,"UPLAND, CA",United States,NaN,NaN,CCB,CABLE,...,2.0,0.0,VMC,Approach,Probable Cause,NaN,2.0,0.0,0.0,Piper PA-28R-200


In [187]:
# calculate percentage of missing values
(df.isna().sum() * 100/len(df)).sort_values(ascending=False)

Schedule                  88.214216
Air.carrier               82.237876
FAR.Description           67.001276
Aircraft.Category         66.774249
Longitude                 59.051640
Latitude                  59.040595
Airport.Code              42.872570
Airport.Name              40.409631
Broad.phase.of.flight     31.485618
pct_fatal_and_serious     17.423424
Total.People              16.982869
Publication.Date          16.030581
Total.Serious.Injuries    14.516248
Total.Minor.Injuries      13.838847
Total.Fatal.Injuries      13.165129
Purpose.of.flight         11.377135
Engine.Type                9.412429
Total.Uninjured            7.059935
Report.Status              6.400942
Number.of.Engines          5.982476
Weather.Condition          4.928333
Aircraft.damage            1.871441
is_destroyed               1.871441
Registration.Number        1.342529
Injury.Severity            0.397605
Country                    0.240526
Amateur.Built              0.068722
Location                   0

In [191]:
# drop coliumns that have more than 50% missing values
drop_cols=["Schedule",                 
"Air.carrier" ,             
"FAR.Description",         
"Aircraft.Category" ,       
"Longitude" ,             
"Latitude"]   

In [192]:
# drop columns

df.drop(columns=drop_cols)

,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Airport.Code,Airport.Name,Injury.Severity,Aircraft.damage,...,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date,Total.People,pct_fatal_and_serious,is_destroyed,Plane.Type
3600,20001214X42040,Accident,LAX83LA093,1983-01-01,"ARROYO GRANDE, CA",United States,NaN,NaN,Non-Fatal,NaN,...,0.0,1.0,VMC,Landing,Probable Cause,NaN,2.0,50.0,NaN,Piccard AX-6
3601,20001214X42095,Accident,SEA83LA036,1983-01-01,"NEWPORT, OR",United States,ONP,NEWPORT MUNICIPAL,Non-Fatal,Substantial,...,1.0,3.0,VMC,Approach,Probable Cause,NaN,4.0,0.0,0.0,Cessna 182P
3602,20001214X42067,Accident,MKC83LA056,1983-01-01,"WOODBINE, IA",United States,3YR,MUNICIPAL,Non-Fatal,Substantial,...,0.0,2.0,VMC,Landing,Probable Cause,NaN,2.0,0.0,0.0,Cessna 182RG
3603,20001214X42063,Accident,MKC83LA050,1983-01-01,"MARYVILLE, MO",United States,78Y,RANKIN,Non-Fatal,Substantial,...,0.0,1.0,VMC,Takeoff,Probable Cause,NaN,1.0,0.0,0.0,Cessna 182P
3604,20001214X42018,Accident,LAX83FUG11,1983-01-01,"UPLAND, CA",United States,CCB,CABLE,Non-Fatal,Substantial,...,2.0,0.0,VMC,Approach,Probable Cause,NaN,2.0,0.0,0.0,Piper PA-28R-200
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
88884,20221227106491,Accident,ERA23LA093,2022-12-26,"Annapolis, MD",United States,NaN,NaN,Minor,NaN,...,0.0,0.0,NaN,NaN,NaN,29-12-2022,1.0,100.0,NaN,Piper PA-28-151
88885,20221227106494,Accident,ERA23LA095,2022-12-26,"Hampton, NH",United States,NaN,NaN,NaN,NaN,...,0.0,0.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,Bellanca 7ECA
88886,20221227106497,Accident,WPR23LA075,2022-12-26,"Payson, AZ",United States,PAN,PAYSON,Non-Fatal,Substantial,...,0.0,1.0,VMC,NaN,NaN,27-12-2022,1.0,0.0,0.0,American Champion Aircraft 8GCBC
88887,20221227106498,Accident,WPR23LA076,2022-12-26,"Morgan, UT",United States,NaN,NaN,NaN,NaN,...,0.0,0.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,Cessna 210N


In [193]:
# confirm columns have been dropped
df.columns

Index(['Event.Id', 'Investigation.Type', 'Accident.Number', 'Event.Date',
       'Location', 'Country', 'Latitude', 'Longitude', 'Airport.Code',
       'Airport.Name', 'Injury.Severity', 'Aircraft.damage',
       'Aircraft.Category', 'Registration.Number', 'Make', 'Model',
       'Amateur.Built', 'Number.of.Engines', 'Engine.Type', 'FAR.Description',
       'Schedule', 'Purpose.of.flight', 'Air.carrier', 'Total.Fatal.Injuries',
       'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured',
       'Weather.Condition', 'Broad.phase.of.flight', 'Report.Status',
       'Publication.Date', 'Total.People', 'pct_fatal_and_serious',
       'is_destroyed', 'Plane.Type'],
      dtype='str')

#### decision
Dropped columns that have more than 50% data missing since imputing it is like inventing data

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [194]:
df.to_csv("./data/cleaned_aviation_data.csv",index=False)